# Delhi/NCR Rental Market Intelligence - Value-For-Money Score Model
This notebook builds the customized value-for-money (VFM) model for our renter segments (Students, Working Professionals, Working Bachelors, Families) to help renters find the best price-to-benefit ratio.
### Methodology:
1. Normalize price, space, safety, amenities, and connectivity components to a 0-100 scale.
2. Set customizable weights representing each segment's preference profile.
3. Calculate aggregate scores (0-100) for every property.
4. Analyze best-value localities.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/rental_listings_cleaned.csv')
print(f'Cleaned listings loaded: {df.shape[0]}')


Cleaned listings loaded: 838


### 1. Scoring Component Functions
We calculate the normalized score matrices for all components.


In [1]:
# Affordability component (lower rent is higher score)
max_rent = df['monthly_rent'].max()
df['score_affordability'] = 100 - (df['monthly_rent'] / max_rent * 100)

# Proximity decay scores
df['score_metro'] = 100 * np.exp(-0.8 * df['metro_distance_km'])
df['score_college'] = 100 * np.exp(-0.8 * df['college_distance_km'])
df['score_office'] = 100 * np.exp(-0.8 * df['office_distance_km'])
df['score_school'] = 100 * np.exp(-0.8 * df['school_distance_km'])
df['score_hospital'] = 100 * np.exp(-0.8 * df['hospital_distance_km'])

# Space score
max_area = df['area_sqft'].max()
df['score_space'] = (df['area_sqft'] / max_area) * 100

# Safety score
zone_safety = df['zone'].map({
    'South Delhi': 85, 'Dwarka': 80, 'Gurugram': 80, 'Noida': 75,
    'North Delhi': 70, 'West Delhi': 70, 'Central Delhi': 70,
    'East Delhi': 65, 'Greater Noida': 65, 'Ghaziabad': 60
}).fillna(70)
cctv_num = df['cctv'].map({'Yes': 10, 'No': 0})
sec_num = df['security'].map({'Yes': 10, 'No': 0})
df['score_safety'] = (zone_safety + cctv_num + sec_num).clip(upper=100)

# Amenities score
amenity_cols = ['ac', 'wifi', 'power_backup', 'lift', 'parking', 'balcony', 'laundry', 'gym', 'food', 'housekeeping']
amenity_count = df[amenity_cols].map(lambda x: 1 if x == 'Yes' else 0).sum(axis=1)
df['score_amenities'] = (amenity_count / len(amenity_cols)) * 100

df[['score_affordability', 'score_metro', 'score_space', 'score_safety', 'score_amenities']].describe()


### 2. Computing Weighted VFM Scores
Apply the weights configuration to get the segment-specific Value Scores.


In [1]:
# Weights configurations
weights = {
    'student': {'affordability': 0.30, 'metro': 0.25, 'college': 0.20, 'amenities': 0.15, 'safety': 0.10},
    'professional': {'metro': 0.25, 'office': 0.25, 'affordability': 0.20, 'amenities': 0.15, 'safety': 0.15},
    'bachelor': {'affordability': 0.30, 'metro': 0.25, 'office': 0.15, 'amenities': 0.15, 'safety': 0.15},
    'family': {'affordability': 0.20, 'space': 0.15, 'school': 0.20, 'hospital': 0.15, 'safety': 0.20, 'metro_office': 0.10}
}

df['vfm_student'] = (df['score_affordability'] * weights['student']['affordability'] +
                      df['score_metro'] * weights['student']['metro'] +
                      df['score_college'] * weights['student']['college'] +
                      df['score_amenities'] * weights['student']['amenities'] +
                      df['score_safety'] * weights['student']['safety']).round(1)

df['vfm_professional'] = (df['score_metro'] * weights['professional']['metro'] +
                           df['score_office'] * weights['professional']['office'] +
                           df['score_affordability'] * weights['professional']['affordability'] +
                           df['score_amenities'] * weights['professional']['amenities'] +
                           df['score_safety'] * weights['professional']['safety']).round(1)

df['vfm_bachelor'] = (df['score_affordability'] * weights['bachelor']['affordability'] +
                       df['score_metro'] * weights['bachelor']['metro'] +
                       df['score_office'] * weights['bachelor']['office'] +
                       df['score_amenities'] * weights['bachelor']['amenities'] +
                       df['score_safety'] * weights['bachelor']['safety']).round(1)

df['vfm_family'] = (df['score_affordability'] * weights['family']['affordability'] +
                     df['score_space'] * weights['family']['space'] +
                     df['score_school'] * weights['family']['school'] +
                     df['score_hospital'] * weights['family']['hospital'] +
                     df['score_safety'] * weights['family']['safety'] +
                     ((df['score_metro'] + df['score_office'])/2) * weights['family']['metro_office']).round(1)

df[['vfm_student', 'vfm_professional', 'vfm_bachelor', 'vfm_family']].agg(['mean', 'median', 'min', 'max'])


### 3. Analyzing Top Localities by Segment
Which localities offer the best VFM for each segment?


In [1]:
print('Top 3 Localities for Students:')
print(df.groupby('locality')['vfm_student'].median().sort_values(ascending=False).head(3))

print('\nTop 3 Localities for Families:')
print(df.groupby('locality')['vfm_family'].median().sort_values(ascending=False).head(3))


Top 3 Localities for Students:
locality
Knowledge Park     79.8
Pari Chowk        76.5
Laxmi Nagar       74.2
Name: vfm_student, dtype: float64

Top 3 Localities for Families:
locality
Dwarka         68.6
Indirapuram    64.9
Vaishali       64.1
Name: vfm_family, dtype: float64
